# longer runs (reruns)

**How to use this notebook:** each person opens their own copy in their own Colab session, with **Runtime > Change runtime type > T4 GPU**, and runs **one job only** (A, B or C below). A fresh Colab session forgets everything, so every job starts with its own setup cell.

| Job | What | Approx. time (from our first runs) |
|---|---|---|
| A | DiffMM, 30 epochs, one gap setting (change `RATIO`) | about 25 to 30 min |
| B | HSTGMatch with its default epochs (50 pre-train + 100 train) | about 40 min |

Keep the browser tab open while a job runs. Everything is copied to Google Drive at the end of each job.

## Job A: DiffMM, longer training

Person 1 uses `RATIO = 0.1`, person 2 uses `0.025`. If you have spare people, use `0.2` and `0.05` as well (these give the full four-point curve).

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/cm4604_cp3
!nvidia-smi | head -12
!git clone https://github.com/decisionintelligence/DiffMM.git
!pip install -q rtree geopandas networkx einops tqdm

Mounted at /content/drive
Wed Nov 11 15:06:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

In [2]:
%%time
RATIO = 0.1      # change to 0.025, 0.2 or 0.05
EPOCHS = 30
!cd DiffMM && python main.py --city porto --keep_ratio {RATIO} --epochs {EPOCHS} --batch_size 128 --gpu_id 0 --train_flag --test_flag 2>&1 | tee ../diffmm_r{RATIO}_30ep.txt

Use GPU: cuda 0
Preparing Data...
edge Num:  11491
node Num:  5330
valid edge Num:  11491
long length:  11653.178220574457 234
width length:  5220.884118697607 105
----- construct cnn graph ------
traj num: 100%|██████████| 10000/10000 [00:30<00:00, 329.18it/s]
training dataset shape: 16000
validation dataset shape: 10000
traj num: 100%|██████████| 10000/10000 [00:30<00:00, 329.05it/s]
testing dataset shape: 10000
Finish data preparing.
modelModelAllShortCut(
  (encoder): TrajEncoder(
    (road_emb): Sequential(
      (0): Linear(in_features=265, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): Norm()
    )
    (point_encoder): PointEncoder(
      (fc_point): Linear(in_features=3, out_features=256, bias=True)
      (transformer): TransformerEncoder(
        (pe): PositionalEncoder()
        (layers): ModuleList(
          (0-1): 2 x EncoderLayer(
            (norm_1): Norm()
            (attn): MultiHeadAttention(

In [3]:
!grep -h 'test_accuracy' diffmm_r*_30ep.txt
!cp diffmm_r*_30ep.txt /content/drive/MyDrive/cm4604_cp3/

test_accuracy: 0.8578292737817738, test_recall: 0.8601580735930736, test_precision: 0.8569305988455987, test_f1: 0.8579956866035582


**What to look for:** `valid_accuracy` should keep rising and then flatten (the code stops early when it stops improving). Write down the final `test_accuracy` line, and the time printed by the `%%time` cell.

## Job B: HSTGMatch, default training length

Before running: upload **porto_to_hstgmatch.py** using the file panel (folder icon on the left).

In [4]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/cm4604_cp3
!nvidia-smi | head -12
!git clone https://github.com/decisionintelligence/DiffMM.git
!git clone https://github.com/Nerooo-g/HSTGMatch.git
!pip install -q torch_geometric pyyaml scikit-learn
!python porto_to_hstgmatch.py --porto DiffMM/data/porto --out HSTGMatch/data/porto_r0.1 --keep_ratio 0.1 --n_train 6000 --n_extra 2000

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Wed Nov 11 15:43:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   74C    P8             16W /   70W |       0MiB /  15360MiB |      0%      Default |
|          

In [5]:
%%time
# stage 1: pre-training, default 50 epochs (about 8 minutes)
!cd HSTGMatch && python scripts/pretrain.py --config configs/default.yaml --data_dir data/porto_r0.1 --batch_size 32 2>&1 | tee ../hstg_pretrain_full.txt

Using device: cuda
Loading data from data/porto_r0.1...
Traceback (most recent call last):
  File "/content/HSTGMatch/scripts/pretrain.py", line 335, in <module>
    main()
    ~~~~^^
  File "/content/HSTGMatch/scripts/pretrain.py", line 225, in main
    preprocessor = Preprocessor.from_files(
        traj_path, road_path,
        cell_size_m=cell_size_m,
        distance_threshold_m=dist_thresh,
    )
  File "/content/HSTGMatch/src/data/preprocess.py", line 187, in from_files
    with open(trajectories_path, "r") as f:
         ~~~~^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'data/porto_r0.1/trajectories.json'
CPU times: user 6.99 ms, sys: 3.86 ms, total: 10.9 ms
Wall time: 3.43 s


In [6]:
%%time
# stage 2: supervised training, default 100 epochs (about 30 minutes)
!cd HSTGMatch && python scripts/train.py --config configs/default.yaml --data_dir data/porto_r0.1 --pretrained_encoder checkpoints/pretrain/best_encoder.pt --batch_size 32 2>&1 | tee ../hstg_train_full.txt

Using device: cuda
Loading data from data/porto_r0.1...
Traceback (most recent call last):
  File "/content/HSTGMatch/scripts/train.py", line 487, in <module>
    main()
    ~~~~^^
  File "/content/HSTGMatch/scripts/train.py", line 296, in main
    with open(traj_path) as f:
         ~~~~^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'data/porto_r0.1/trajectories.json'
CPU times: user 13 ms, sys: 43 µs, total: 13 ms
Wall time: 4.35 s


In [7]:
!cp hstg_*_full.txt /content/drive/MyDrive/cm4604_cp3/
!cp -r HSTGMatch/checkpoints /content/drive/MyDrive/cm4604_cp3/hstg_checkpoints

**Decision rule for Job B:** watch the `F1=` value in the training log. If it is still below about 0.05 after roughly 30 epochs, stop the run (Runtime > Interrupt) and report honestly that HSTGMatch could not be reproduced at our data scale. Do not spend more than this one session on it.